# Gemma 4 31B — Language-U V12 Atlas-Guided Cortical Cascade (A100 / G4 / L4 High-RAM)

- **Atlas-Guided 1-Bit / 2-Bit Swap (`~6.259 GiB <= 6.28 GiB`, smaller than V10):** Moves wasted 2-bit bytes off dead upper layers (`L50–53, 56–57`) directly onto the empirical Cognitive Core (`L26, 27, 28, 29, 32, 41, 46` full `2b_gauss`, plus `L20, 24, 33, 36, 43` `2b_gauss down_proj`), while keeping 45 body MLPs in fast 1-bit `1b_rm`.
- **Core-Protected + Null-Bypass Dual E2E KL QAT:** Protects all 16 Cognitive Cores at 100% density while training downstream layers to absorb the 4 Tier-2 bypass skips (`L16, 18, 40, 45`) seamlessly and sharpening `L29` semantic outputs for `VA Mirror` 4 KiB neighborhood pages.
- **Outputs saved & SHA-256 verified on Google Drive (`/content/drive/MyDrive/gemma4_v12_atlas_cascade`) + `drive.flush_and_unmount()` + `runtime.unassign()` auto-shutdown.**

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch

# 1. Instant Hardware Envelope Guard (before downloading anything)
assert torch.cuda.is_available(), (
    "No CUDA GPU detected! Go to Runtime -> Change runtime type -> select A100 GPU (or L4 GPU) + High-RAM, then click Run all."
)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
gpu_name = torch.cuda.get_device_name(0)
meminfo_kb = int([l.split()[1] for l in Path("/proc/meminfo").read_text().splitlines() if l.startswith("MemTotal:")][0])
sys_ram_gb = meminfo_kb / (1024**2)
print(f"[Hardware Check] GPU={gpu_name} ({vram_gb:.2f} GiB VRAM) | System RAM={sys_ram_gb:.1f} GiB", flush=True)
assert vram_gb >= 20.0, (
    f"Connected to {gpu_name} ({vram_gb:.1f} GiB VRAM), which is too small for 31B QAT. "
    "Go to Runtime -> Change runtime type -> select A100 GPU (recommended) or L4 GPU + High-RAM, then click Run all."
)
assert sys_ram_gb >= 25.0, (
    f"System RAM is {sys_ram_gb:.1f} GiB (Standard VM). Turn ON the 'High-RAM' toggle in Runtime -> Change runtime type, then click Run all."
)

# 2. Mount Google Drive & Sync Latest Public V12 Repo
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

REPO_URL = "https://github.com/DannyB-bit/Language-U-Gemma4.git"
REPO_DIR = Path("/content/Language-U-Gemma4")
DRIVE_V12 = Path("/content/drive/MyDrive/gemma4_v12_atlas_cascade")
DRIVE_V12.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-b", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)

print("[Repo Commit]", subprocess.check_output(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], text=True).strip(), flush=True)

# 3. Ensure Gemma 4 Dependencies (>=5.0.0) & Run 6-Gate V12 Self-Test
need_pip = False
try:
    import datasets, safetensors, tokenizers, huggingface_hub, PIL
    from transformers.models.gemma4.modeling_gemma4 import Gemma4VisionModel
except Exception:
    need_pip = True
if need_pip:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers>=5.0.0", "accelerate", "safetensors", "huggingface_hub", "datasets", "tokenizers", "pillow"],
        check=True,
    )

SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v12_atlas_cascade_gpu.py"
res = subprocess.run([sys.executable, str(SCRIPT), "--selftest"], text=True, capture_output=True)
print(res.stdout, res.stderr, flush=True)
res.check_returncode()

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/Language-U-Gemma4")
SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v12_atlas_cascade_gpu.py"
LOG_PATH = Path("/content/v12_train.log")
DRIVE_V12 = Path("/content/drive/MyDrive/gemma4_v12_atlas_cascade")

env = os.environ.copy()
env["V12_DRIVE_DIR"] = str(DRIVE_V12)
env["V12_OUT_DIR"] = "/content/v12_outputs"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
env["COLAB_AUTO_UNASSIGN"] = "0"
env["PYTHONUNBUFFERED"] = "1"

with open(LOG_PATH, "w", encoding="utf-8") as log_f:
    proc = subprocess.Popen(
        [sys.executable, "-u", str(SCRIPT), "--run", "--e2e-steps", "200"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
        log_f.write(line)
    rc = proc.wait()

shutil.copy2(LOG_PATH, DRIVE_V12 / "v12_train.log")
subprocess.run(["sync"], check=False)
assert rc == 0, f"V12 pipeline exited with {rc} (log saved to {DRIVE_V12 / 'v12_train.log'})"
print(subprocess.check_output(["sha256sum", "-c", "SHA256SUMS_V12.txt"], cwd=str(DRIVE_V12), text=True), flush=True)
print((DRIVE_V12 / "v12_report.json").read_text(encoding="utf-8"), flush=True)
print("[Zero-CU Guard] All V12 SHA-256 checksums verified on Drive; flushing Drive & auto-disconnecting runtime...", flush=True)
from google.colab import drive, runtime
try:
    drive.flush_and_unmount()
except Exception as e:
    print(f"[Drive Flush Note] {e}", flush=True)
runtime.unassign()